In [31]:
import pandas as pd

In [32]:
df = pd.read_csv("SBAnational.csv", dtype=str, 
                 low_memory=False)
print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")

Total rows: 899164
Total columns: 27


In [33]:
df.head()

,LoanNr_ChkDgt,Name,City,State,Zip,Bank,BankState,NAICS,ApprovalDate,ApprovalFY,...,RevLineCr,LowDoc,ChgOffDate,DisbursementDate,DisbursementGross,BalanceGross,MIS_Status,ChgOffPrinGr,GrAppv,SBA_Appv
0,1000014003,ABC HOBBYCRAFT,EVANSVILLE,IN,47711,FIFTH THIRD BANK,OH,451120,28-Feb-97,1997,...,N,Y,NaN,28-Feb-99,"$60,000.00",$0.00,P I F,$0.00,"$60,000.00","$48,000.00"
1,1000024006,LANDMARK BAR & GRILLE (THE),NEW PARIS,IN,46526,1ST SOURCE BANK,IN,722410,28-Feb-97,1997,...,N,Y,NaN,31-May-97,"$40,000.00",$0.00,P I F,$0.00,"$40,000.00","$32,000.00"
2,1000034009,"WHITLOCK DDS, TODD M.",BLOOMINGTON,IN,47401,GRANT COUNTY STATE BANK,IN,621210,28-Feb-97,1997,...,N,N,NaN,31-Dec-97,"$287,000.00",$0.00,P I F,$0.00,"$287,000.00","$215,250.00"
3,1000044001,"BIG BUCKS PAWN & JEWELRY, LLC",BROKEN ARROW,OK,74012,1ST NATL BK & TR CO OF BROKEN,OK,0,28-Feb-97,1997,...,N,Y,NaN,30-Jun-97,"$35,000.00",$0.00,P I F,$0.00,"$35,000.00","$28,000.00"
4,1000054004,"ANASTASIA CONFECTIONS, INC.",ORLANDO,FL,32801,FLORIDA BUS. DEVEL CORP,FL,0,28-Feb-97,1997,...,N,N,NaN,14-May-97,"$229,000.00",$0.00,P I F,$0.00,"$229,000.00","$229,000.00"


In [34]:
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print(f"\nColumn names:")
print(df.columns.tolist())

Rows: 899164
Columns: 27

Column names:
['LoanNr_ChkDgt', 'Name', 'City', 'State', 'Zip', 'Bank', 'BankState', 'NAICS', 'ApprovalDate', 'ApprovalFY', 'Term', 'NoEmp', 'NewExist', 'CreateJob', 'RetainedJob', 'FranchiseCode', 'UrbanRural', 'RevLineCr', 'LowDoc', 'ChgOffDate', 'DisbursementDate', 'DisbursementGross', 'BalanceGross', 'MIS_Status', 'ChgOffPrinGr', 'GrAppv', 'SBA_Appv']


In [35]:
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})

print(missing_df[missing_df['Missing Count'] > 0])

                  Missing Count  Missing %
Name                         14       0.00
City                         30       0.00
State                        14       0.00
Bank                       1559       0.17
BankState                  1566       0.17
NewExist                    136       0.02
RevLineCr                  4528       0.50
LowDoc                     2582       0.29
ChgOffDate               736465      81.91
DisbursementDate           2368       0.26
MIS_Status                 1997       0.22


In [36]:
print("=== LOAN STATUS ===")
print(df['MIS_Status'].value_counts(dropna=False))

=== LOAN STATUS ===
MIS_Status
P I F     739609
CHGOFF    157558
NaN         1997
Name: count, dtype: int64


In [37]:
before = len(df)
df = df[df['MIS_Status'].notna() & (df['MIS_Status'].str.strip() != '')]
after = len(df)

print(f"Before: {before} rows")
print(f"After:  {after} rows")
print(f"Remove: {before - after} blank rows ✅")

Before: 899164 rows
After:  897167 rows
Remove: 1997 blank rows ✅


In [38]:
before = len(df)
df = df[df['NewExist'] != '0']
after = len(df)

print(f"Before: {before} rows")
print(f"After:  {after} rows")  
print(f"Remove: {before - after} undefined rows ✅")

Before: 897167 rows
After:  896139 rows
Remove: 1028 undefined rows ✅


In [39]:
money_columns = ['GrAppv', 'SBA_Appv', 'DisbursementGross', 'BalanceGross', 'ChgOffPrinGr']

for col in money_columns:
    df[col] = df[col].str.replace('$', '', regex=False)
    df[col] = df[col].str.replace(',', '', regex=False)
    df[col] = df[col].str.strip()

print("Money columns cleanned!")
print("\nSample GrAppv values:")
print(df['GrAppv'].head(5).tolist())

Money columns cleanned!

Sample GrAppv values:
['60000.00', '40000.00', '287000.00', '35000.00', '229000.00']


In [40]:
def clean_revlinecr(val):
    if val in ['Y', 'N']:
        return val
    else:
        return 'Other'

df['RevLineCr'] = df['RevLineCr'].apply(clean_revlinecr)

print("RevLineCr unique values:")
print(df['RevLineCr'].value_counts())

RevLineCr unique values:
RevLineCr
N        418322
Other    277189
Y        200628
Name: count, dtype: int64


In [41]:
def clean_lowdoc(val):
    if val in ['Y', 'N']:
        return val
    else:
        return 'Other'

df['LowDoc'] = df['LowDoc'].apply(clean_lowdoc)

print("LowDoc unique values:")
print(df['LowDoc'].value_counts())

LowDoc unique values:
LowDoc
N        780099
Y        110046
Other      5994
Name: count, dtype: int64


In [42]:
df = df.fillna('')

print("All NaN values are replace with empty string")

All NaN values are replace with empty string


In [43]:
print("=== FINAL REPORT ===")
print(f"Total clean rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")

print("\n=== MIS_Status (target) ===")
print(df['MIS_Status'].value_counts())

print("\n=== RevLineCr ===")
print(df['RevLineCr'].value_counts())

print("\n=== LowDoc ===")
print(df['LowDoc'].value_counts())

print("\n=== GrAppv sample ===")
print(df['GrAppv'].head(5).tolist())

print("\nReady to Go!")

=== FINAL REPORT ===
Total clean rows: 896139
Total columns: 27

=== MIS_Status (target) ===
MIS_Status
P I F     738644
CHGOFF    157495
Name: count, dtype: int64

=== RevLineCr ===
RevLineCr
N        418322
Other    277189
Y        200628
Name: count, dtype: int64

=== LowDoc ===
LowDoc
N        780099
Y        110046
Other      5994
Name: count, dtype: int64

=== GrAppv sample ===
['60000.00', '40000.00', '287000.00', '35000.00', '229000.00']

Ready to Go!


In [44]:
output_path = "Downloads/SBAnational_clean.csv"

df.to_csv(output_path, index=False)

print(f"✅ Clean file saved!")
print(f"Location: {output_path}")
print(f"Total rows saved: {len(df)}")

✅ Clean file saved!
Location: Downloads/SBAnational_clean.csv
Total rows saved: 896139
